# Signal stability — performance signals (X) across the season

*Read-only informative artifact. Characterises how each candidate signal's
distribution moves across the season so a human can decide which signals are
poolable and which must be read within-block. No gate decisions, no
PROCEED/STOP verdict. The stability verdicts are **operational heuristics**
(fixed normalised-shift thresholds) offered as analytical guidance — not
statistical tests, not a gate.*

## Questions an analyst asks of the signals over time

- **Do signals drift across the season?** Does a signal's typical level or
  spread move from the early third to the late third?
- **Is a signal as trustworthy for a GW36 decision as for a GW10 one?** A
  signal whose distribution shifts late cannot be read off a season-pooled
  fingerprint at the end of the campaign.
- **Which signals can be pooled across the season, and which should be read
  within-block?** Every other foundation notebook pools the season; this one
  asks, per signal and position, whether that pooling is safe.

The structure `signal.ipynb` answered *what each signal looks like at rest*
(season-pooled). This notebook asks *how that fingerprint moves across the
season* — the same read-only class of artifact, one axis further.

## Setup

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and the **participation** population (`minutes > 0`), carve the season into
three contiguous GW **blocks** (early / mid / late), and define the candidate
signal set.

This is a *descriptive characterisation* notebook, so it uses the full season —
no early-GW lower bound (that GW-6 cut in the older EDA-1 record was a
*predictive-evaluation* choice, not relevant here).

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. `minutes` can be NULL for some rows; `minutes > 0`
naturally excludes those. The 60-minute performance-boundary question is
**deferred to the `population/` layer** and is deliberately not baked in here.

**Block definition.** The temporal unit is a *third of the season*. We split
`GW STUDY_GW_MIN .. STUDY_GW_MAX` into three near-equal contiguous blocks,
computed dynamically from the live GW range. The exact bounds are printed by
the setup cell below.

**Double-gameweek (DGW) honesty note.** Most candidate signals are **additive
per fixture** (`xg`, `xa`, `xgi`, `bps`, `bonus`, the ICT components), so a DGW
row roughly **doubles** them. DGW rounds cluster **late** in the season, so a
signal can look like it "drifts up" late purely because more rows in the late
block carry two fixtures. These block distributions **pool SGW and DGW rows**
(no normalisation, no exclusion); per-fixture treatment is **deferred to the
`fixture/` layer**. Read late-block shifts with that confound in mind.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.block_distributions import (
    compute_signal_block_distributions,
)
from research.kernels.diagnostic.stability import (
    assess_distribution_stability,
    resolve_pooling_strategy,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that GW-6 cut in the older EDA-1
# record was a predictive-evaluation choice, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. `minutes` can be NULL
# for some rows; minutes > 0 naturally excludes NULLs (NULL comparisons are
# False). The 60-minute performance boundary is NOT imposed here -- that
# question is deferred to the population/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]

# Temporal unit: split the WHOLE completed season into three contiguous,
# (near-)equal GW blocks -- early / mid / late -- computed dynamically from the
# GW range so this notebook adapts to whatever the season's data_cutoff_gw is.
def _make_thirds(gw_min, gw_max):
    span = gw_max - gw_min + 1
    cut1 = gw_min + span // 3            # end of "early" (inclusive)
    cut2 = gw_min + (2 * span) // 3      # end of "mid"   (inclusive)
    return {
        "early": (gw_min, cut1),
        "mid":   (cut1 + 1, cut2),
        "late":  (cut2 + 1, gw_max),
    }

GW_BLOCKS = _make_thirds(STUDY_GW_MIN, STUDY_GW_MAX)
BLOCK_ORDER = ["early", "mid", "late"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Population: minutes > 0 (participation, not a performance gate), n = {len(df):,} player-gameweeks")
for pos in POSITIONS:
    print(f"  {pos}: {len(df[df.position == pos]):>6,}")
print("\nGW blocks (thirds of the completed season, inclusive bounds):")
for name in BLOCK_ORDER:
    lo, hi = GW_BLOCKS[name]
    print(f"  {name:<6} GW {lo:>2} - {hi:<2}")

In [ ]:
# Candidate signals: same numeric per-gameweek performance signals the
# structure signal.ipynb chose (read from dal/mart/mart_schema.py + the live
# mart). Rolling/contextual and identity/market/structural columns are excluded
# -- those are not raw weekly performance signals.
SIGNALS = [
    "xg", "xa", "xgi", "xgc",
    "ict_index", "influence", "creativity", "threat",
    "bps", "bonus",
]
SIGNALS = [s for s in SIGNALS if s in df.columns]
print("Candidate signals:", SIGNALS)

## (a) Per-signal, per-block distribution by position

**What we measure** — for every (signal, position) pair, the median and IQR of
the signal in each season third, via `compute_signal_block_distributions` over
the candidate signal list. One row per (signal, position, block); `n` per cell
is shown so thin blocks (degenerate signal/position combinations, e.g. a
keeper's `xg`) are visible as NaN distribution stats.

*Stat glossary for this table:* **mean** — arithmetic average, pulled up by rare hauls; **median** — middle value, the robust "typical" return; **std** — typical distance from the mean in points, sensitive to outliers; **IQR (p75−p25)** — spread of the middle 50%, robust to skew; **p90/p99** — score exceeded by only 10%/1% of appearances (practical ceiling); **skew** — positive = long right tail of rare high scores.

**What it means** — this is each signal's per-position fingerprint, read three
times across the season. A median that climbs late, or an IQR that swells,
tells the analyst the signal is *not stationary* there — so a model that reads
that signal off a season-pooled level will mis-rate late-season players.

**What it doesn't mean** — these are **block-pooled** distributions across
players, not a within-player signal trajectory (deferred). Block boundaries are
arbitrary thirds. Additive signals inherit the **DGW confound**: DGW rounds
cluster late, so a late-block rise in `xg`/`bps`/etc. can be fixture-doubling
rather than a genuine distributional shift. Drift here is *spread/level* over
time, not predictive value for `total_points` (out of scope for this layer).

**Guiding question** — *Do signals drift across the season, and is a signal as
trustworthy late as it is early?*

In [ ]:
# Per-(signal, position, block) median/IQR. NaN distribution stats appear where
# a block has n < 10 (e.g. structurally degenerate signal/position pairs).
sig_blocks = compute_signal_block_distributions(
    df, SIGNALS, POSITIONS, gw_column="gw", gw_blocks=GW_BLOCKS,
)
sig_blocks["block"] = pd.Categorical(sig_blocks["block"], categories=BLOCK_ORDER, ordered=True)
sig_blocks["position"] = pd.Categorical(sig_blocks["position"], categories=POSITIONS, ordered=True)
sig_blocks["signal"] = pd.Categorical(sig_blocks["signal"], categories=SIGNALS, ordered=True)
sig_blocks = sig_blocks.sort_values(["signal", "position", "block"]).reset_index(drop=True)
display(sig_blocks[["signal", "position", "block", "min_gw", "max_gw", "n", "median", "iqr"]])

In [ ]:
# A compact median-by-block pivot makes drift legible across all signals at
# once: read each (signal x position) row left-to-right (early -> mid -> late).
median_pivot = sig_blocks.pivot_table(
    index=["signal", "position"], columns="block", values="median", observed=True,
)[BLOCK_ORDER]
display(median_pivot.round(3))

## (b) Stability verdict and pooling guidance, per (signal, position)

**What we measure** — for each (signal, position) pair we feed its three-block
stats to `assess_distribution_stability` → `{stable, moderate_shift,
unstable}` (largest normalised median shift between blocks), then map to a
pooling decision via `resolve_pooling_strategy` (`pool_confirmed` /
`pool_with_caveat` / `restrict_to_midseason`). Results are laid out as a
signal × position grid of verdicts.

**What it means** — this is the per-cell answer to "is this signal poolable
across the season at this position?" — `pool_confirmed` cells are safe to read
off a season-pooled fingerprint; `restrict_to_midseason` cells are where a
season-pooled level would mislead a late-season decision and the signal should
be read within-block.

**What it doesn't mean** — the thresholds (`STABLE_THRESHOLD = 0.5`,
`UNSTABLE_THRESHOLD = 1.5` on the normalised median shift) are **operational
heuristics**, not statistical tests; the grid is **analytical guidance, not a
gate**. Pairs with fewer than two valid blocks (degenerate signal/position
combinations) default to `stable`/`unstable` by the kernel's small-sample rule
— read those against the `n` column in (a), not as a real stability claim. The
DGW confound applies: late-clustered doubling can manufacture an `unstable`
verdict on additive signals.

**Guiding question** — *Which signals can be pooled across the season, and
which should be read within-block — and does that depend on position?*

In [ ]:
# Per-(signal, position) verdict + pooling guidance. Heuristic thresholds
# (STABLE=0.5, UNSTABLE=1.5 on normalised median shift) -- analytical guidance,
# NOT a gate. Pairs with < 2 valid blocks fall back to the kernel's
# small-sample rule; cross-reference n in (a).
rows = []
for sig in SIGNALS:
    for pos in POSITIONS:
        block_stats = sig_blocks[(sig_blocks["signal"] == sig) & (sig_blocks["position"] == pos)]
        verdict = assess_distribution_stability(block_stats)
        pooling = resolve_pooling_strategy(verdict)
        rows.append({
            "signal": sig,
            "position": pos,
            "n_blocks_valid": int(block_stats["median"].notna().sum()),
            "stability_verdict": verdict,
            "pooling_guidance": pooling,
        })
signal_stability = pd.DataFrame(rows)
display(signal_stability)

In [ ]:
# Verdict grid as a heatmap reveals whether instability clusters by signal or by
# position -- a pattern the long table hides. Encode the 3 verdicts as 0/1/2.
VERDICT_CODE = {"stable": 0, "moderate_shift": 1, "unstable": 2}
grid = (
    signal_stability
    .assign(code=signal_stability["stability_verdict"].map(VERDICT_CODE))
    .pivot(index="signal", columns="position", values="code")
    .reindex(index=SIGNALS, columns=POSITIONS)
)
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap(["#2ca02c", "#ff7f0e", "#d62728"])  # stable / moderate / unstable
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(grid.values.astype(float), cmap=cmap, norm=norm, aspect="auto")
ax.set_xticks(range(len(POSITIONS)))
ax.set_xticklabels(POSITIONS)
ax.set_yticks(range(len(grid.index)))
ax.set_yticklabels(grid.index)
label_for = {0: "stab", 1: "mod", 2: "unst"}
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, label_for[int(v)], ha="center", va="center",
                    color="white", fontsize=7)
ax.set_title("Signal stability verdict by signal x position\n(heuristic; green=stable, orange=moderate, red=unstable)", fontsize=10)
plt.tight_layout()
plt.show()

## What the signals' season-stability looks like

Plain-language summary of the descriptive picture (not a verdict, not a gate):

- The block table, the median-by-block pivot, and the verdict grid show, per
  signal and position, whether the signal's typical level and spread move
  across the early / mid / late thirds.
- The per-cell **stability verdict** and **pooling guidance** translate that
  into a read on which (signal, position) pairs are safe to pool across the
  season and which should be read within-block — but only as a **heuristic**
  (fixed normalised-shift thresholds), offered as guidance.
- Degenerate signal/position pairs (a keeper's `xg`, a forward's `xgc`) show
  thin or NaN blocks in (a) and fall back to the kernel's small-sample verdict
  rule — read those against the `n` column, not as real stability claims.
- Late-block movement on **additive** signals should be read against the **DGW
  confound**: double-gameweeks cluster late and roughly double those signals,
  so apparent late drift can be fixture-doubling.

All figures are **whole-season**, over the **participation** population
(`minutes > 0` — not a performance gate; the 60-minute boundary is deferred to
the `population/` layer). Thresholds are operational heuristics, not
statistical tests. Within-player signal dynamics are deferred to the future
`temporal` treatments; per-fixture DGW normalisation is deferred to the
`fixture/` layer.